# Will it be an early Spring?

On February 2<sup>nd</sup> every year Punxsutawney Phil makes a prediction about if there will be an early Spring or if Winter will continue for 6 more weeks (till about mid-March). He is however not very accurate (well, according to [The Inner Circle](https://www.groundhog.org/inner-circle) he is 100% correct but the human handler may not interpret his response correctly). The overall goal is to be able to predict if it will be an early Spring.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help). To schedule an appointment, look at the client's Google Calendar and then reach out to them to propose a time. I estimate the first meeting will need about 20 minutes. Come prepared.

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
   * Must provide a function `generate_samples(years: list[int], phil_preds: list[bool]) -> pd.DataFrame` that takes a list of years, a list of `True`/`False`s for Phil's prediction for those years, and gives back a data frame for those years matching the formatting of the train and test sets.
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes a the train or test set (or anything returned by `generate_samples()`) and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preperation should be noted here but actually implemented in the Preperation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit all data files and a pickled final model along with this notebook.

# Frame the Problem and Look at the Big Picture

1. **Define the objective in business terms** \
The client wants to predict whether planting conditions in Western Pennsylvania will be suitable during the first 10 days of May. An early Spring is defined by four criteria: soil temperature above 50°F, minimum air temperature above 30°F, average air temperature above 40°F, and no snow on the ground. The goal is to develop a machine learning model that predicts whether all four criteria will be met, allowing farms to make accurate planting decisions.
2. **How will the solution be used?** \
The model will be used to make a binary prediction of whether the following May will have an early Spring. The prediction must be available by March 1 when farms order their seeds so it can help them plan for the upcoming planting season. Its output will be a yes/no prediction that can support the client's planting decisions.
3. **What are the current solutions or workarounds?** \
Currently, Punxsutawney Phil's prediction provides an informal indication of whether winter will continue or Spring will arrive early. However, these predictions cover a different time period from the client's planting criteria, and it is not expected to be a reliable indicator of the target. The client also has historical soil and weather records that can be used to identify patterns in past planting conditions.
4. **How should you frame this problem?** \
This is a supervised, binary classification problem because the model will learn from historical examples with known early Spring outcomes and predict either True or False for a future year. It will use offline learning since the model will be trained and evaluated using historical data rather than continuously updating itself as new data arrives. The model will be used for forecasting, with all input features restricted to information available by March 1 of the year being predicted.
5. **How should performance be measured?** \
The primary performance metric should reflect the relative costs of false positive and false negative predictions for the farms. Predicting no early Spring when an early Spring occurs is much more costly for the farms, so recall for the early-Spring class should be emphasized. The model should also use precision and F1 score so that both types of errors can be evaluated. The classes are imbalanced, sp accuracy would not be sufficient.
6. **What would be the minimum performance needed to reach the business objective?** \
The model must produce an expected financial return greater than the cost of planting. A harvest is expected to generate approximately five times the amount spent on seeds, so the model should perform better than the 5 to 1 benchmark when its predictions are used.
7. **What are comparable problems? Can you reuse experience or tools?** \
8. **Is human expertise available?** \
The client is available to clarify the planting criteria, explain the provided datasets, and provide additional information about the farms and their historical records. The farms can supply historical soil temperature data and other relevant records. We can use this to clarify assumptions and assess whether the model's predictions are useful for the business objective.
9. **How would you solve the problem manually?** \
Examine historical soil and weather records to look for patterns associated with years in which all the planting criteria were met. We would look at winter temperatures, soil conditions, and broader climate patterns that could be related to conditions in early May. We would then use information available by March 1 to estimate whether the upcoming May would meet the planting criteria.
10. **List the assumptions you or others have made so far. Verify assumptions if possible.** \

# Get the Data

### Data sources
**Provided by client:**
- early_spring.csv: Historical early Spring labels.
- phil_pred.csv: Punxsutawney Phil's historical predictions.
- soil.csv: Soil-related measurements, including temperature and moisture.
- weather.csv: Weather observations from farms in Western Pennsylvania.

**Additional sources:**
- anom_monthly.csv: Monthly Nino 3.4 sea-surface-temperature anomalies from the NOAA Climate Prediction Center. These values help identify El Nino and La Nina conditions.
- nao_monthly.csv: Monthly North Atlantic Oscillation (NAO) index values from the NOAA Climate Prediction Center. These values describe the phase and strength of the NAO, a climate pattern that influences winter weather in the North Atlantic region.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

def load_early_spring():
    early_spring = pd.read_csv(
        "early_spring.csv",
        dtype={"year": "int64", "early_spring": "boolean"},
    )
    return early_spring

def load_phil_pred():
    phil_pred = pd.read_csv(
        "phil_pred.csv",
        dtype={"year": "int64", "prediction": "boolean"},
    )
    return phil_pred

def load_soil():
    soil = pd.read_csv("soil.csv", parse_dates=["day"])
    return soil

def load_weather():
    weather = pd.read_csv("weather.csv", parse_dates=["date"])
    return weather

def summarize_weather_codes(weather):
    """Count special codes"""
    columns = [
        "max_temp", "min_temp", "avg_temp",
        "precipitation", "snowfall", "snowdepth",
    ]

    for column in columns:
        values = weather[column].astype("string").str.strip()

        print(f"\n{column}")
        print("Special values:")
        print(values[values.isin(["T", "M"])].value_counts())

def compare_year_coverage(early_spring, phil_pred, soil, weather, anom_data, nao_data):
    """Compare the years represented in each dataset."""
    years_by_dataset = {
        "early_spring": set(early_spring["year"].dropna()),
        "phil_pred": set(phil_pred["year"].dropna()),
        "soil": set(soil["day"].dt.year.dropna()),
        "weather": set(weather["date"].dt.year.dropna()),
        "anom_data": set(load_anom()["year"].dropna()),
        "nao_data": set(load_nao()["year"].dropna())
    }

    for name, years in years_by_dataset.items():
        print(
            f"{name}: {len(years)} years, "
            f"{min(years)}–{max(years)}"
        )

def inspect_date_coverage(data, date_col, name):
    dates = data[date_col]

    print(f"\n{name}")
    print("Observations by month:")
    print(dates.dt.month.value_counts().sort_index())

    print("\nObservations per month and year:")
    monthly_counts = (
        data.assign(year=dates.dt.year, month=dates.dt.month)
        .groupby(["year", "month"])
        .size()
        .unstack(fill_value=0)
    )
    print(monthly_counts.tail(5))

def inspect_non_datetime_coverage(data, col, name):
    print(f"\n{name}")
    print("Observations by month:")
    print(data[col].value_counts().sort_index())

    print("\nObservations per month and year:")
    monthly_counts = (
        data.assign(year=data["year"], month=data["month"])
        .groupby(["year", "month"])
        .size()
        .unstack(fill_value=0)
    )
    print(monthly_counts.tail(5))

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ANOM_URL = "https://www.cpc.ncep.noaa.gov/data/indices/Rnino34.ascii.txt"
ANOM_CACHE = DATA_DIR / "anom_monthly.csv"

def load_anom() -> pd.DataFrame:
    if ANOM_CACHE.exists():
        return pd.read_csv(ANOM_CACHE)
    df = pd.read_csv(ANOM_URL, sep=r"\s+")
    df.columns = ["year", "month", "anom"]
    df.to_csv(ANOM_CACHE, index=False)
    return df

NAO_URL = "https://www.cpc.ncep.noaa.gov/products/precip/CWlink/pna/norm.nao.monthly.b5001.current.ascii"
NAO_CACHE = DATA_DIR / "nao_monthly.csv"

def load_nao() -> pd.DataFrame:
    if NAO_CACHE.exists():
        return pd.read_csv(NAO_CACHE)
    df = pd.read_csv(NAO_URL, sep=r"\s+", names=["year", "month","nao"])
    df.to_csv(NAO_CACHE, index=False)
    return df

In [2]:
early_spring, phil_pred, soil, weather = load_early_spring(), load_phil_pred(), load_soil(), load_weather()

In [3]:
early_spring

,year,early_spring
0,1992,False
1,2003,False
2,1978,False
3,1954,False
4,1972,False
...,...,...
67,1991,True
68,2020,False
69,1950,False
70,1956,False


In [4]:
phil_pred

,year,prediction
0,1947,False
1,1948,False
2,1949,False
3,1950,True
4,1951,False
...,...,...
67,2022,False
68,2023,False
69,2024,True
70,2025,False


In [5]:
soil

,day,mean_soil_temp,mean_soil_moisture,min_soil_temp,min_soil_moisture,max_soil_temp,max_soil_moisture
0,1947-01-01,-3.179167,0.477208,-4.0,0.475,-1.3,0.480
1,1947-01-02,-2.500000,0.497083,-3.1,0.477,-1.8,0.518
2,1947-01-03,-1.766667,0.514833,-2.6,0.504,-0.4,0.519
3,1947-01-04,0.750000,0.497542,-0.1,0.490,2.2,0.511
4,1947-01-05,0.237500,0.486000,-0.7,0.484,1.1,0.489
...,...,...,...,...,...,...,...
4333,2026-02-25,-0.287500,0.479917,-0.6,0.473,0.2,0.505
4334,2026-02-26,-0.270833,0.488083,-0.5,0.483,-0.1,0.497
4335,2026-02-27,-0.320833,0.482125,-0.7,0.479,0.2,0.489
4336,2026-02-28,1.066667,0.478250,0.1,0.460,5.3,0.487


In [6]:
weather

,date,max_temp,min_temp,avg_temp,precipitation,snowfall,snowdepth
0,1947-01-01,30.0,24.0,27.0,0.12,0.5,1.0
1,1947-01-02,44.0,29.0,36.5,0.49,0.0,0.0
2,1947-01-03,54.0,41.0,47.5,0.04,0.0,M
3,1947-01-04,42.0,24.0,33.0,0.0,0.0,M
4,1947-01-05,33.0,25.0,29.0,0.0,0.0,M
...,...,...,...,...,...,...,...
4333,2026-02-25,37.0,17.0,27.0,0.2,2.5,3.0
4334,2026-02-26,45.0,22.0,33.5,T,T,0.0
4335,2026-02-27,42.0,21.0,31.5,0.0,0.0,0.0
4336,2026-02-28,53.0,24.0,38.5,0.0,0.0,0.0


In [7]:
# Weather special values
summarize_weather_codes(weather)


max_temp
Special values:
max_temp
M    89
Name: count, dtype: int64[pyarrow]

min_temp
Special values:
min_temp
M    83
Name: count, dtype: int64[pyarrow]

avg_temp
Special values:
avg_temp
M    104
Name: count, dtype: int64[pyarrow]

precipitation
Special values:
precipitation
T    692
M     16
Name: count, dtype: int64[pyarrow]

snowfall
Special values:
snowfall
T    711
M     44
Name: count, dtype: int64[pyarrow]

snowdepth
Special values:
snowdepth
T    770
M     75
Name: count, dtype: int64[pyarrow]


In [8]:
compare_year_coverage(
    early_spring, phil_pred, soil, weather
)

early_spring: 72 years, 1947–2026
phil_pred: 72 years, 1947–2026
soil: 72 years, 1947–2026
weather: 72 years, 1947–2026


In [9]:
inspect_date_coverage(soil, "day", "Soil")
inspect_date_coverage(weather, "date", "Weather")

soil_dates = set(soil["day"])
weather_dates = set(weather["date"])


Soil
Observations by month:
day
1    2232
2    2034
3      72
Name: count, dtype: int64

Observations per month and year:
month   1   2  3
year            
2022   31  28  1
2023   31  28  1
2024   31  29  1
2025   31  28  1
2026   31  28  1

Weather
Observations by month:
date
1    2232
2    2034
3      72
Name: count, dtype: int64

Observations per month and year:
month   1   2  3
year            
2022   31  28  1
2023   31  28  1
2024   31  29  1
2025   31  28  1
2026   31  28  1


In [10]:
# list the years that are missing from each dataset
print("Missing years in weather dataset:", pd.date_range(weather["date"].min(), weather["date"].max()).year.difference(weather["date"].dt.year).unique())
print("Missing years in soil dataset:", pd.date_range(soil["day"].min(), soil["day"].max()).year.difference(soil["day"].dt.year).unique())
print("Missing years in early_spring dataset:", pd.date_range(early_spring["year"].min(), early_spring["year"].max()).year.difference(early_spring["year"]).unique())
print("Missing years in phil_pred dataset:", pd.date_range(phil_pred["year"].min(), phil_pred["year"].max()).year.difference(phil_pred["year"]).unique())

Missing years in weather dataset: Index([1957, 1964, 1979, 1990, 1993, 1996, 2007, 2011], dtype='int32')
Missing years in soil dataset: Index([1957, 1964, 1979, 1990, 1993, 1996, 2007, 2011], dtype='int32')
Missing years in early_spring dataset: Index([], dtype='int32')
Missing years in phil_pred dataset: Index([], dtype='int32')


### Initial Dataset Evaluation

The four datasets provided by the client cover the same 72 years, from 1947 through 2026. The annual label and Phil prediction datasets each contain one observation per year, while the soil and weather datasets contain 4,338 daily observations each. All four datasets are missing the same 8 years (1957, 1964, 1979, 1990, 1993, 1996, 2007, 2011).

The target variable is somewhat imbalanced. 75% of the years are labeled as not having an early Spring, and 25% are labeled as having an early Spring. As a result, a model that always predicts no early Spring would achieve 75% accuracy. This baseline should be considered when evaluating the classification models, and additional metrics will be needed to assess how well they identify early Spring years.

The weather dataset contains special values that are not currently recognized as missing. Temperature fields contain `M` values, and precipitation, snowfall, and snow depth contain both `M` and `T` values. These require further investigation before preprocessing because missing measurements and trace amounts should not automatically be treated as zero.

Although the datasets cover the same years, the daily date coverage still needs to be examined to determine which months are represented and whether the soil and weather observations align exactly. The next steps are to verify this coverage, confirm that predictors are available by the March 1 deadline, and determine how to combine the daily measurements into annual features without introducing information that would not be available at prediction time.

The soil and weather datasets cover January 1 through March 1 for each year from 1947 to 2026. Both contain 4,338 observations and have identical date coverage, with no duplicate dates or unmatched observations between the datasets. The number of observations per year varies between 60 and 61 because of leap years. These datasets contain different units of measurement for temperature. The weather dataset uses fahrenheit and the soil dataset uses celsius. One of these should be converted in order to easily compare with each other.

In [16]:
anom_data, nao_data = load_anom(), load_nao()

In [17]:
anom_data

,year,month,anom
0,1949,12,-1.03
1,1950,1,-1.34
2,1950,2,-1.19
3,1950,3,-0.70
4,1950,4,-0.99
...,...,...,...
917,2026,5,0.52
918,2026,6,1.07
919,2026,7,1.33
920,2026,8,1.67


In [18]:
nao_data

,year,month,nao
0,1950,1,0.9200
1,1950,2,0.4000
2,1950,3,-0.3600
3,1950,4,0.7300
4,1950,5,-0.5900
...,...,...,...
916,2026,5,-0.7418
917,2026,6,0.1014
918,2026,7,-0.3134
919,2026,8,-1.9164


In [31]:
compare_year_coverage(early_spring, phil_pred, soil, weather, anom_data, nao_data)

early_spring: 72 years, 1947–2026
phil_pred: 72 years, 1947–2026
soil: 72 years, 1947–2026
weather: 72 years, 1947–2026
anom_data: 78 years, 1949–2026
nao_data: 77 years, 1950–2026


In [35]:
inspect_non_datetime_coverage(anom_data, "month", "Anomaly Data")


Anomaly Data
Observations by month:
month
1     77
2     77
3     77
4     77
5     77
6     77
7     77
8     77
9     77
10    76
11    76
12    77
Name: count, dtype: int64

Observations per month and year:
month  1   2   3   4   5   6   7   8   9   10  11  12
year                                                 
2022    1   1   1   1   1   1   1   1   1   1   1   1
2023    1   1   1   1   1   1   1   1   1   1   1   1
2024    1   1   1   1   1   1   1   1   1   1   1   1
2025    1   1   1   1   1   1   1   1   1   1   1   1
2026    1   1   1   1   1   1   1   1   1   0   0   0


In [37]:
inspect_non_datetime_coverage(nao_data, "month", "NAO Data")


NAO Data
Observations by month:
month
1     77
2     77
3     77
4     77
5     77
6     77
7     77
8     77
9     77
10    76
11    76
12    76
Name: count, dtype: int64

Observations per month and year:
month  1   2   3   4   5   6   7   8   9   10  11  12
year                                                 
2022    1   1   1   1   1   1   1   1   1   1   1   1
2023    1   1   1   1   1   1   1   1   1   1   1   1
2024    1   1   1   1   1   1   1   1   1   1   1   1
2025    1   1   1   1   1   1   1   1   1   1   1   1
2026    1   1   1   1   1   1   1   1   1   0   0   0


### Initial Evaluation of Additional Datasets

The two additional datasets contain monthly climate information. The anom_data dataset contains Nino 3.4 temperature anomalies. These values show whether the sea-surface temperature is above or below its usual average. For this evaluation, values from -1 to -0.5 indicate La Nina, values from -0.5 to 0.5 indicate neutral conditions, and values from 0.5 to 1 indicate El Nino.

The nao_data dataset contains the North Atlantic Oscillation index. Positive values show the positive NAO phase, which usually means stronger winds and storms farther north. Negative values show the negative NAO phase, which usually means weaker winds and storms farther south. Values near zero indicate a weak or neutral phase.

Both datasets provide complete monthly coverage for 1950 through 2025. The anomaly dataset also includes December 1949, and they both include January through September 2026. These partial years should be handled when creating annual features.

Because predictions must be available by March 1, features derived from these datasets must use only months available by that date. The 2026 partial records also need handling when deciding the final training and test years.

The soil and weather datasets store each observation as a full date, while the other datasets either use a separate numeric year column or both numeric year and month columns. This difference will need to be handled when comparing the datasets.